Dataset Source: FUNSD (Form Understanding in Noisy Scanned Documents) -  https://guillaumejaume.github.io/FUNSD/ 

In [1]:
# Download Dependencies and Dataset
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install jiwer -q
!wget -q -nc https://guillaumejaume.github.io/FUNSD/dataset.zip
!unzip -q -o dataset.zip
print("Setup Complete!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.9 MB/s eta 0:00:00a 0

In [2]:
# Start Ollama Server in Background
import os, time, subprocess

!pkill -9 ollama
time.sleep(2)

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

models = ["glm-ocr", "qwen2.5vl:3b", "llava:13b", "moondream"]
for m in models:
    print(f"Downloading {m}...")
    !ollama pull {m} > /dev/null 2>&1

print("All models ready!")

All models ready!


In [3]:
!pip install ollama

In [4]:
import os, json, base64, re
from collections import Counter
from ollama import Client
import jiwer

client = Client(host='http://127.0.0.1:11434')

def clean_repeated_lines(text):
    seen = set()
    lines = []
    for line in text.splitlines():
        clean = re.sub(r'[^a-z0-9\s]', '', line.lower()).strip()
        if not clean:
            continue
        if len(clean.split()) >= 3 and clean in seen:
            break
        seen.add(clean)
        lines.append(line)
    return "\n".join(lines)

def evaluate_model(model_name):
    test_images_dir = "dataset/testing_data/images/"
    test_ann_dir = "dataset/testing_data/annotations/"
    test_files = sorted(os.listdir(test_images_dir))

    total_cer_acc = 0
    total_recall = 0
    total_f1 = 0
    print(f"Starting FUNSD Evaluation ({model_name}) on {len(test_files)} forms...\n")

    for idx, filename in enumerate(test_files, 1):
        img_path = os.path.join(test_images_dir, filename)
        with open(img_path, "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode("utf-8")
            
        ann_path = os.path.join(test_ann_dir, filename.replace(".png", ".json"))
        with open(ann_path, "r") as f:
            ann_data = json.load(f)
            
        sorted_items = sorted(ann_data["form"], key=lambda item: (item["box"][1] // 20, item["box"][0]))
        ground_truth_text = " ".join([item["text"] for item in sorted_items if item["text"].strip()])
        
        print(f"[{idx}/{len(test_files)}] Analyzing {filename}...")
        
        response = client.chat(
            model=model_name,
            messages=[{
                'role': 'user',
                'content': 'Extract all the text from this image. Output only the raw text, with no introduction or explanation.',
                'images': [img_b64]
            }],
            options={
                'temperature': 0.0,
                'num_ctx': 4096,
                'num_predict': 600,
                'repeat_penalty': 1.2,
                'stop': ['---', '```']
            }
        )
        
        raw_predicted = response.get('message', {}).get('content', '').strip()
        predicted_text = clean_repeated_lines(raw_predicted)
        
        clean_gt = re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', ' ', ground_truth_text.lower())).strip()
        clean_pred = re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', ' ', predicted_text.lower())).strip()
        
        cer = jiwer.cer(clean_gt, clean_pred)
        cer_acc = max(0, (1 - cer) * 100)
        
        gt_words = clean_gt.split()
        pred_words = clean_pred.split()
        common = Counter(gt_words) & Counter(pred_words)
        num_same = sum(common.values())
        
        precision = (num_same / len(pred_words) * 100) if pred_words else 0
        recall = (num_same / len(gt_words) * 100) if gt_words else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        print(f"Words Read: {len(pred_words)} (Ground Truth: {len(gt_words)})")
        print(f"CER Accuracy: {cer_acc:.2f}% | Word Recall: {recall:.2f}% | Word F1-Score: {f1:.2f}%")
        print("-" * 50)
        
        total_cer_acc += cer_acc
        total_recall += recall
        total_f1 += f1

    avg_cer_acc = total_cer_acc / len(test_files)
    avg_recall = total_recall / len(test_files)
    avg_f1 = total_f1 / len(test_files)

    print("==================================================")
    print(f"MODEL                      : {model_name}")
    print(f"FINAL AVERAGE CER ACCURACY : {avg_cer_acc:.2f}%")
    print(f"FINAL AVERAGE WORD RECALL  : {avg_recall:.2f}%")
    print(f"FINAL AVERAGE WORD F1-SCORE: {avg_f1:.2f}%")
    print("==================================================")

In [5]:
evaluate_model('glm-ocr')

Starting FUNSD Evaluation (glm-ocr) on 50 forms...

[1/50] Analyzing 82092117.png...
Words Read: 219 (Ground Truth: 222)
CER Accuracy: 98.22% | Word Recall: 97.30% | Word F1-Score: 97.96%
--------------------------------------------------
[2/50] Analyzing 82200067_0069.png...
Words Read: 166 (Ground Truth: 153)
CER Accuracy: 59.82% | Word Recall: 94.12% | Word F1-Score: 90.28%
--------------------------------------------------
[3/50] Analyzing 82250337_0338.png...
Words Read: 216 (Ground Truth: 214)
CER Accuracy: 91.41% | Word Recall: 98.13% | Word F1-Score: 97.67%
--------------------------------------------------
[4/50] Analyzing 82251504.png...
Words Read: 203 (Ground Truth: 224)
CER Accuracy: 79.80% | Word Recall: 85.71% | Word F1-Score: 89.93%
--------------------------------------------------
[5/50] Analyzing 82252956_2958.png...
Words Read: 95 (Ground Truth: 114)
CER Accuracy: 76.74% | Word Recall: 81.58% | Word F1-Score: 89.00%
--------------------------------------------------

In [6]:
evaluate_model('qwen2.5vl:3b')

Starting FUNSD Evaluation (qwen2.5vl:3b) on 50 forms...

[1/50] Analyzing 82092117.png...
Words Read: 223 (Ground Truth: 222)
CER Accuracy: 97.75% | Word Recall: 97.75% | Word F1-Score: 97.53%
--------------------------------------------------
[2/50] Analyzing 82200067_0069.png...
Words Read: 150 (Ground Truth: 153)
CER Accuracy: 85.18% | Word Recall: 91.50% | Word F1-Score: 92.41%
--------------------------------------------------
[3/50] Analyzing 82250337_0338.png...
Words Read: 214 (Ground Truth: 214)
CER Accuracy: 92.20% | Word Recall: 96.73% | Word F1-Score: 96.73%
--------------------------------------------------
[4/50] Analyzing 82251504.png...
Words Read: 10 (Ground Truth: 224)
CER Accuracy: 3.14% | Word Recall: 3.57% | Word F1-Score: 6.84%
--------------------------------------------------
[5/50] Analyzing 82252956_2958.png...
Words Read: 106 (Ground Truth: 114)
CER Accuracy: 84.65% | Word Recall: 91.23% | Word F1-Score: 94.55%
------------------------------------------------

In [7]:
evaluate_model('llava:13b')

Starting FUNSD Evaluation (llava:13b) on 50 forms...

[1/50] Analyzing 82092117.png...
Words Read: 28 (Ground Truth: 222)
CER Accuracy: 12.01% | Word Recall: 5.41% | Word F1-Score: 9.60%
--------------------------------------------------
[2/50] Analyzing 82200067_0069.png...
Words Read: 23 (Ground Truth: 153)
CER Accuracy: 12.09% | Word Recall: 3.27% | Word F1-Score: 5.68%
--------------------------------------------------
[3/50] Analyzing 82250337_0338.png...
Words Read: 135 (Ground Truth: 214)
CER Accuracy: 29.24% | Word Recall: 17.29% | Word F1-Score: 21.20%
--------------------------------------------------
[4/50] Analyzing 82251504.png...
Words Read: 9 (Ground Truth: 224)
CER Accuracy: 3.62% | Word Recall: 2.23% | Word F1-Score: 4.29%
--------------------------------------------------
[5/50] Analyzing 82252956_2958.png...
Words Read: 36 (Ground Truth: 114)
CER Accuracy: 19.94% | Word Recall: 7.02% | Word F1-Score: 10.67%
--------------------------------------------------
[6/50] An

In [8]:
evaluate_model('moondream')

Starting FUNSD Evaluation (moondream) on 50 forms...

[1/50] Analyzing 82092117.png...
Words Read: 8 (Ground Truth: 222)
CER Accuracy: 1.32% | Word Recall: 0.45% | Word F1-Score: 0.87%
--------------------------------------------------
[2/50] Analyzing 82200067_0069.png...
Words Read: 8 (Ground Truth: 153)
CER Accuracy: 2.34% | Word Recall: 1.31% | Word F1-Score: 2.48%
--------------------------------------------------
[3/50] Analyzing 82250337_0338.png...
Words Read: 8 (Ground Truth: 214)
CER Accuracy: 1.10% | Word Recall: 0.47% | Word F1-Score: 0.90%
--------------------------------------------------
[4/50] Analyzing 82251504.png...
Words Read: 62 (Ground Truth: 224)
CER Accuracy: 19.81% | Word Recall: 11.16% | Word F1-Score: 17.48%
--------------------------------------------------
[5/50] Analyzing 82252956_2958.png...
Words Read: 8 (Ground Truth: 114)
CER Accuracy: 2.22% | Word Recall: 0.88% | Word F1-Score: 1.64%
--------------------------------------------------
[6/50] Analyzing 